# RT Paper 3 All-Benchmarks Runner

Use this notebook to fetch, normalize, and run the Paper 3 quick benchmark loop from one Colab workflow.

Auto-supported benchmark downloads:

- `msc_train`, `msc_valid`, `msc_test`
- `locomo10`
- `longmemeval_s_cleaned`, `longmemeval_m_cleaned`

Manual-source fallback benchmarks in the same notebook:

- `gapchat_manual`
- `realtalk_manual`
- `evolmem_manual`
- `normalized_manual`

Recommended quick order:

1. `msc_valid`
2. `locomo10`
3. `longmemeval_s_cleaned`


In [ ]:
REPO_URL = "https://github.com/SteveMama/rt-geometry-memory.git"
REPO_DIR = "/content/rt-geometry-memory"
BATCH_PREFIX = "paper3_bench_v1"

BENCHMARK_NAME = "msc_valid"  # msc_train, msc_valid, msc_test, locomo10, longmemeval_s_cleaned, longmemeval_m_cleaned, gapchat_manual, realtalk_manual, evolmem_manual, normalized_manual
MANUAL_SOURCE_PATH = "/content/manual_benchmark_source.json"
MANUAL_FORMAT = "normalized"  # normalized, msc, locomo, longmemeval
RUN_ALL_AUTO_BENCHMARKS = False
AUTO_BENCHMARKS = ["msc_valid", "locomo10", "longmemeval_s_cleaned"]

MODEL_KEYS = "qwen25_15b"
BUDGETS = "0.20,0.35,0.50"
POLICIES = "uniform,semantic,geometry,geometry_keep_compress_drop"
LIMIT_CONVERSATIONS = 24
TARGET_TURN_STRIDE = 4
MAX_TARGET_TURNS = 16


In [ ]:
%cd /content
!rm -rf $REPO_DIR
!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!bash scripts/colab_setup.sh


## Benchmark registry

This cell resolves fetch paths and normalization modes. For `GapChat`, `REALTALK`, and `EvolMem`, provide one source file path in `MANUAL_SOURCE_PATH` and set `MANUAL_FORMAT` to the matching raw schema.


In [ ]:
from pathlib import Path
import os
import subprocess

AUTO_BENCHMARK_CONFIG = {
    "msc_train": {"format": "msc", "source": "/content/msc_train.jsonl"},
    "msc_valid": {"format": "msc", "source": "/content/msc_valid.jsonl"},
    "msc_test": {"format": "msc", "source": "/content/msc_test.jsonl"},
    "locomo10": {"format": "locomo", "source": "/content/locomo10.json"},
    "longmemeval_s_cleaned": {"format": "longmemeval", "source": "/content/longmemeval_s_cleaned.json"},
    "longmemeval_m_cleaned": {"format": "longmemeval", "source": "/content/longmemeval_m_cleaned.json"},
}
MANUAL_BENCHMARKS = {"gapchat_manual", "realtalk_manual", "evolmem_manual", "normalized_manual"}


def resolve_source(name: str):
    if name in AUTO_BENCHMARK_CONFIG:
        config = AUTO_BENCHMARK_CONFIG[name]
        return config["source"], config["format"]
    if name in MANUAL_BENCHMARKS:
        return MANUAL_SOURCE_PATH, MANUAL_FORMAT
    raise ValueError(f"Unsupported benchmark name: {name}")


SELECTED_SOURCE, SELECTED_FORMAT = resolve_source(BENCHMARK_NAME)
SELECTED_JSONL = f"{REPO_DIR}/benchmarks/{BENCHMARK_NAME}_normalized.jsonl"
print(
    {
        "benchmark": BENCHMARK_NAME,
        "source": SELECTED_SOURCE,
        "format": SELECTED_FORMAT,
        "jsonl": SELECTED_JSONL,
    }
)


## Fetch selected benchmark

Auto-supported benchmarks download directly. Manual benchmarks reuse `MANUAL_SOURCE_PATH`.


In [ ]:
if BENCHMARK_NAME in AUTO_BENCHMARK_CONFIG:
    fetch_cmd = [
        "python",
        "scripts/download_public_benchmark.py",
        "--benchmark",
        BENCHMARK_NAME,
        "--output",
        SELECTED_SOURCE,
    ]
    print("RUN", " ".join(fetch_cmd))
    subprocess.run(fetch_cmd, check=True, cwd=REPO_DIR)
else:
    if not os.path.exists(SELECTED_SOURCE):
        raise FileNotFoundError(
            f"Manual benchmark source not found: {SELECTED_SOURCE}. Upload or mount it first."
        )
    print(f"Using manual benchmark source: {SELECTED_SOURCE}")


## Normalize selected benchmark into RT JSONL


In [ ]:
normalize_cmd = [
    "python",
    "scripts/prepare_public_benchmark_jsonl.py",
    "--format",
    SELECTED_FORMAT,
    "--input",
    SELECTED_SOURCE,
    "--output",
    SELECTED_JSONL,
    "--family",
    BENCHMARK_NAME,
]
print("RUN", " ".join(normalize_cmd))
subprocess.run(normalize_cmd, check=True, cwd=REPO_DIR)


## Run selected benchmark


In [ ]:
run_cmd = [
    "bash",
    "scripts/run_paper3_quick_benchmark.sh",
    f"{BATCH_PREFIX}_{BENCHMARK_NAME}",
    SELECTED_JSONL,
    MODEL_KEYS,
    BUDGETS,
    POLICIES,
    str(LIMIT_CONVERSATIONS),
    str(TARGET_TURN_STRIDE),
    str(MAX_TARGET_TURNS),
]
print("RUN", " ".join(run_cmd))
subprocess.run(run_cmd, check=True, cwd=REPO_DIR)


## Inspect selected benchmark reports


In [ ]:
study_dir = Path(REPO_DIR) / "results" / "paper3" / "studies" / f"{BATCH_PREFIX}_{BENCHMARK_NAME}"
print(study_dir)
for report_name in ["study_report.md", "pairwise_report.md"]:
    report_path = study_dir / report_name
    if report_path.exists():
        print(f"\n===== {report_name} =====\n")
        print(report_path.read_text(encoding="utf-8")[:12000])


## Optional: run all auto-supported benchmarks

This loop runs the configured `AUTO_BENCHMARKS` sequentially with the same policies and bounded defaults. It skips manual-source benchmarks.


In [ ]:
if not RUN_ALL_AUTO_BENCHMARKS:
    print("Skipping full auto-benchmark loop. Set RUN_ALL_AUTO_BENCHMARKS = True to enable it.")
else:
    for benchmark_name in AUTO_BENCHMARKS:
        config = AUTO_BENCHMARK_CONFIG[benchmark_name]
        source_path = config["source"]
        normalized_path = f"{REPO_DIR}/benchmarks/{benchmark_name}_normalized.jsonl"
        fetch_cmd = [
            "python",
            "scripts/download_public_benchmark.py",
            "--benchmark",
            benchmark_name,
            "--output",
            source_path,
        ]
        normalize_cmd = [
            "python",
            "scripts/prepare_public_benchmark_jsonl.py",
            "--format",
            config["format"],
            "--input",
            source_path,
            "--output",
            normalized_path,
            "--family",
            benchmark_name,
        ]
        run_cmd = [
            "bash",
            "scripts/run_paper3_quick_benchmark.sh",
            f"{BATCH_PREFIX}_{benchmark_name}",
            normalized_path,
            MODEL_KEYS,
            BUDGETS,
            POLICIES,
            str(LIMIT_CONVERSATIONS),
            str(TARGET_TURN_STRIDE),
            str(MAX_TARGET_TURNS),
        ]
        print(f"\n=== {benchmark_name}: fetch ===")
        subprocess.run(fetch_cmd, check=True, cwd=REPO_DIR)
        print(f"=== {benchmark_name}: normalize ===")
        subprocess.run(normalize_cmd, check=True, cwd=REPO_DIR)
        print(f"=== {benchmark_name}: run ===")
        subprocess.run(run_cmd, check=True, cwd=REPO_DIR)
